# Chapter 10, Application 1: Qubit-efficient (maximum-likelihood) amplitude estimation

Companion notebook to *Quantum Algorithms from First Principles*, Chapter 10,
"Building maximum-likelihood amplitude estimation" and "Testing the amplitude estimator".

Qubits: exactly what $\mathcal{A}$ needs plus the oracle ancilla. No clock register, and
no controlled $\mathcal{Q}$. Schedule $r_k = 0,1,2,4,\dots,2^M$, $N_{\rm shots}$ per value,
then maximize
$\log L(\theta)=\sum_k\big[h_k\log\sin^2((2r_k+1)\theta)+(N_{\rm shots}-h_k)\log\cos^2((2r_k+1)\theta)\big]$
over $\theta\in(0,\pi/2)$ and report $\hat g=\sin^2\hat\theta$. The test that matters: the
error falls as $1/N_{\rm queries}$, not $1/\sqrt{N_{\rm queries}}$.

$\mathcal{A}$, $\mathcal{S}_{\rm good}$ and $\mathcal{Q}$ are those of the Chapter 6 amplitude
amplification notebook. Convention: book wire $j$ is Qiskit qubit `bq(j)`.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize_scalar
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Statevector
from qiskit_aer import AerSimulator

bq = lambda j, tot: tot - 1 - j
n = 3
tot = n + 1
wires = [bq(j, tot) for j in range(n)]
anc = bq(n, tot)


def A_prep(phi):
    qc = QuantumCircuit(tot, name="A")
    qc.h(wires[0]); qc.h(wires[1]); qc.ry(phi, wires[2])
    return qc


def S_good():
    qc = QuantumCircuit(tot, name="S_good"); qc.mcx(wires, anc); return qc


def S_zero():
    qc = QuantumCircuit(tot, name="S_0")
    qc.x(wires); qc.h(wires[-1]); qc.mcx(wires[:-1], wires[-1]); qc.h(wires[-1]); qc.x(wires)
    return qc


def Q_op(phi):
    qc = QuantumCircuit(tot)
    qc.compose(S_good(), inplace=True)
    qc.compose(A_prep(phi).inverse(), inplace=True)
    qc.compose(S_zero(), inplace=True)
    qc.compose(A_prep(phi), inplace=True)
    return qc

## Test 1: $\mathcal{Q}$ rotates by $2\theta$

In [2]:
phi = 1.2
g_true = 0.25 * np.sin(phi / 2) ** 2
theta_true = np.arcsin(np.sqrt(g_true))
for r in range(5):
    qc = QuantumCircuit(tot); qc.x(anc); qc.h(anc); qc.compose(A_prep(phi), inplace=True)
    for _ in range(r): qc.compose(Q_op(phi), inplace=True)
    p = Statevector(qc).probabilities(wires[::-1])[0b111]
    assert abs(p - np.sin((2 * r + 1) * theta_true) ** 2) < 1e-10
print(f"g = {g_true:.5f}, theta = {theta_true:.5f}; Pr[good] = sin^2((2r+1)theta) for r = 0..4")

g = 0.07971, theta = 0.28621; Pr[good] = sin^2((2r+1)theta) for r = 0..4


## The estimator

In [3]:
sim = AerSimulator()


def run_counts(phi, r, shots, seed):
    """Number of 'good' outcomes (|111>) in `shots` runs of A Q^r."""
    qc = QuantumCircuit(tot, n)
    qc.x(anc); qc.h(anc); qc.compose(A_prep(phi), inplace=True)
    for _ in range(r): qc.compose(Q_op(phi), inplace=True)
    qc.measure(wires, [n - 1 - j for j in range(n)])
    counts = sim.run(transpile(qc, sim), shots=shots, seed_simulator=seed).result().get_counts()
    return counts.get("1" * n, 0)


def mlae(phi, M, shots, seed=0):
    rs = [0] + [2**k for k in range(M + 1)]
    hs = [run_counts(phi, r, shots, seed + i) for i, r in enumerate(rs)]
    eps = 1e-12

    def neg_log_L(theta):
        tot_ = 0.0
        for r, h in zip(rs, hs):
            p = np.clip(np.sin((2 * r + 1) * theta) ** 2, eps, 1 - eps)
            tot_ += h * np.log(p) + (shots - h) * np.log(1 - p)
        return -tot_

    # the likelihood is multimodal: grid first, then refine
    grid = np.linspace(1e-4, np.pi / 2 - 1e-4, 4000)
    t0 = grid[np.argmin([neg_log_L(t) for t in grid])]
    res = minimize_scalar(neg_log_L, bounds=(max(t0 - 0.01, 1e-6), min(t0 + 0.01, np.pi / 2 - 1e-6)), method="bounded")
    n_queries = sum(shots * (2 * r + 1) for r in rs)      # oracle calls: A Q^r costs 2r+1 queries
    return np.sin(res.x) ** 2, n_queries


g_hat, Nq = mlae(phi, M=4, shots=100)
print(f"estimate {g_hat:.5f} vs true {g_true:.5f}, |error| {abs(g_hat-g_true):.2e}, queries {Nq}")

estimate 0.07942 vs true 0.07971, |error| 2.88e-04, queries 6800


## Test 2: error against queries, slope $-1$ (Heisenberg), not $-1/2$ (sampling)

Average the error over several seeds at each $M$, and compare with plain sampling using
the same number of oracle queries.

In [4]:
Ms = [1, 2, 3, 4, 5, 6]
errs, Nqs, errs_classical = [], [], []
rng = np.random.default_rng(0)
for M in Ms:
    e, N = [], None
    for s in range(6):
        g_hat, N = mlae(phi, M, 100, seed=1000 * M + s)
        e.append(abs(g_hat - g_true))
    errs.append(np.sqrt(np.mean(np.square(e)))); Nqs.append(N)
    # classical: N queries of A, each a Bernoulli(g) sample
    errs_classical.append(np.sqrt(g_true * (1 - g_true) / N))
slope = np.polyfit(np.log(Nqs), np.log(errs), 1)[0]
print("queries:", Nqs)
print("RMS error:", np.round(errs, 5))
print(f"fitted slope {slope:.2f} (Heisenberg -1; sampling would be -0.5)")
plt.loglog(Nqs, errs, "o-", label="MLAE")
plt.loglog(Nqs, errs_classical, "s--", label=r"sampling $\sqrt{g(1-g)/N}$")
plt.loglog(Nqs, errs[0] * (np.array(Nqs) / Nqs[0]) ** -1.0, ":", label="slope -1")
plt.xlabel("oracle queries"); plt.ylabel("RMS error in g"); plt.legend(); plt.show()
assert slope < -0.75

queries: [900, 1800, 3500, 6800, 13300, 26200]
RMS error: [0.00371 0.00324 0.0041  0.00052 0.00046 0.00019]
fitted slope -0.97 (Heisenberg -1; sampling would be -0.5)


## Test 3: $g$ near 0, and at the top of this $\mathcal{A}$'s range ($g\le 1/4$ here), expose the likelihood clipping

In [5]:
for phi_ in [0.08, np.pi - 0.05]:
    g_ = 0.25 * np.sin(phi_ / 2) ** 2
    g_hat, N = mlae(phi_, M=5, shots=100, seed=7)
    print(f"g = {g_:.5f}: estimate {g_hat:.5f}, error {abs(g_hat - g_):.1e}")

g = 0.00040: estimate 0.00037, error 2.7e-05


g = 0.24984: estimate 0.24961, error 2.3e-04


With $M=5$ the schedule reaches $r=32$, so the deepest circuit has 65 oracle queries;
the whole run uses a few thousand queries and resolves $g$ to about $10^{-4}$. A sampling
estimator with the same budget is an order of magnitude worse, and the gap widens as
$M$ grows. No clock register was used anywhere.